# Figure 2 — Section 3 phase portrait

Reproduces the 2D phase portrait of the access-pricing dynamics
under the symmetric triangle price function $p(q)$ with identity sensitivity
$f(p)=p$, single price-sensitive class (no inelastic users).

The figure shows:

- the nullclines $\eta_1$ ($\dot q = 0$) and $\eta_2$ ($\dot R = 0$),
- their two intersections $(R_1^*, q_1^*)$ and $(R_2^*, q_2^*)$ (the equilibria),
- the trapping region $R \le R^\dagger$, $q_m \le q \le q_2^*$ (hatched),
- an illustrative trajectory through points $A$–$E$.

Output: `Fig2.pdf`


## 1. Imports and configuration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from functools import partial

from access_pricing.model import (
    mu_trapezoidal, p_triangle, f_identity, alpha_linear,
    derive_alpha_2d_from_two_equilibria,
)
from access_pricing.parameters import FIG2 as P
from access_pricing.plot_style import configure

configure()


## 2. Parameters and derived alpha

In [ ]:
# The alpha slope and intercept are derived from the chosen equilibria q1, q2
# using the joint equilibrium condition with f(p) = p (identity sensitivity).
alpha_slope, alpha_intcp = derive_alpha_2d_from_two_equilibria(
    q1=P.q1, q2=P.q2, K_R=P.K_R, mu_max=P.mu_max,
    q_c=P.q_c, q_max=P.q_max, q_m=P.q_m, beta=P.beta,
)

# Bind parameters into single-argument closures of q only.
mu_fn    = partial(mu_trapezoidal, q_c=P.q_c, q_max=P.q_max, mu_max=P.mu_max)
p_fn     = partial(p_triangle,     q_m=P.q_m, beta=P.beta)
f_fn     = f_identity
alpha_fn = partial(alpha_linear,   slope=alpha_slope, intcp=alpha_intcp, q_max=P.q_max)


## 3. Geometric reference points

The figure uses several derived constants:

- $R^\dagger = K_R / \alpha(0)$ — vertical reference line; bound on the trapping region.
- $q^\dagger$ — q-coordinate where $\eta_1$ crosses $R = R^\dagger$ (closed form below).
- $q^\dagger_2$ — q-coordinate where $\eta_2$ crosses $R = R^\dagger$ (closed form below).

Both $q^\dagger$ values are computed analytically from the nullcline definitions
(in the original notebooks they were hardcoded; here the geometry is explicit).


In [ ]:
# Equilibrium-related quantities
mu_star_1 = float(mu_fn(P.q1))
mu_star_2 = float(mu_fn(P.q2))
p_star_1  = float(p_fn(P.q1))
p_star_2  = float(p_fn(P.q2))

R_star_1 = (P.K_R - mu_star_1) / p_star_1
R_star_2 = mu_star_2 / alpha_fn(P.q2)

# R^dagger: K_R / alpha(0)
R_dag = P.K_R / float(alpha_fn(0.0))

# q^dagger: eta_1(q) = mu(q)/alpha(q) = R_dag, with q < q_c so mu(q) = mu_max*q/q_c.
# Solving (q * mu_max/q_c) = R_dag * (intcp + slope*q):
q_dag = (R_dag * alpha_intcp) / (P.mu_max / P.q_c - R_dag * alpha_slope)

# q^dagger_2: eta_2(q) = K_R / (alpha(q) + f(q)) = R_dag,
# i.e. alpha(q) + f(q) = alpha(0). With f(q) = beta*(2*q_m - q) in [q_m, 2*q_m]
# and alpha(q) = intcp + slope*q: (slope - beta)*q + 2*q_m*beta = 0.
q_dag2 = (2 * P.q_m * P.beta) / (P.beta - alpha_slope)

print(f"R_dag    = {R_dag:.6f}")
print(f"q_dag    = {q_dag:.10f}")
print(f"q_dag2   = {q_dag2:.10f}")


## 4. The figure

In [ ]:
plt.figure(figsize=(5, 3.5))

# --- Reference lines ---
plt.plot([-100, 1500], [P.q_max, P.q_max], ':k', lw=1)
plt.plot([-100, 1500], [P.q_m, P.q_m], ':k', lw=1)
plt.plot([R_dag, R_dag], [-9, q_dag2 + 10], ':k', lw=1)
plt.plot([-100, 1500], [q_dag, q_dag], ':k', lw=1)
plt.text(R_dag - 20, q_dag2 + 11, r'$R^\dagger$')

# --- Nullclines over the full q range ---
qs       = np.linspace(0, P.q_max, 501)
f_q      = np.array([f_fn(p_fn(qq)) for qq in qs])
alpha_q  = np.array([alpha_fn(qq)   for qq in qs])
mu_q     = np.array([mu_fn(qq)      for qq in qs])

plt.plot(mu_q / alpha_q,         qs, c='#EA6849', label=r'$\eta_1$ (i.e., $\dot{q}=0$)')
plt.plot(P.K_R / (alpha_q + f_q), qs, c='#22B6BA', label=r'$\eta_2$ (i.e., $\dot{R}=0$)')

# --- Hatched region (between eta_1 and clip(eta_2, R_dag), over [q_dag, q2]) ---
qs_hatch_h = np.linspace(q_dag, P.q2, 500)
f_qhh      = np.array([f_fn(p_fn(qq)) for qq in qs_hatch_h])
alpha_qhh  = np.array([alpha_fn(qq)   for qq in qs_hatch_h])
mu_qhh     = np.array([mu_fn(qq)      for qq in qs_hatch_h])
eta2_h     = np.maximum(R_dag, P.K_R / (alpha_qhh + f_qhh))

plt.fill_betweenx(
    qs_hatch_h, mu_qhh / alpha_qhh, eta2_h,
    hatch='\\\\\\', edgecolor='#FCBC06',
    facecolor='#EECD64', linewidth=0.0,
)

# --- Region labels ---
plt.text(100, 75, '$\\dot{R}>0$\n$\\dot{q}<0$')
plt.text(680, 25, '$\\dot{R}<0$\n$\\dot{q}>0$')

# --- Equilibria ---
plt.scatter([R_star_1], [P.q1], s=15, c='k', zorder=3)
plt.scatter([R_star_2], [P.q2], s=15, c='k', zorder=3)
plt.text(225, 32, r'$(R_1^*, q_1^*)$', fontsize=8)
plt.text(920, 77, r'$(R_2^*, q_2^*)$', fontsize=8)

# --- Illustrative trajectory through A-B-C-D-E ---
arbR, arbQ = 450, 68
plt.plot(
    [0, 0, P.K_R / (alpha_q[340] + f_q[340]), arbR, arbR, 0],
    [0, arbQ, qs[340], qs[290], 0, 0],
    c='gray', marker='o', markersize=5,
)
plt.plot(
    [P.K_R / (alpha_q[340] + f_q[340]), arbR, arbR],
    [qs[340], arbQ, qs[290]],
    c='gray', ls='--',
)
plt.scatter([arbR], [arbQ], marker='x', s=30, c='b')

# Letter labels
plt.text(-28, -7,        'A', c='gray')
plt.text(-30, arbQ + 3,  'B', c='gray')
plt.text(mu_q[290]/alpha_q[290] + 30, arbQ - 12, 'D', c='gray')
plt.text(arbR - 140, qs[340] + 4, 'C', c='gray')
plt.text(arbR, -7,       'E', c='gray')

# --- Axes ---
plt.xlabel('R'); plt.ylabel('q')
plt.yticks([0, P.q_m, q_dag, 100],
           [0, r'$q_m$', r'$q^\dagger$', r'$q_{max}$'])
plt.xticks([0, P.K_R / (alpha_q[0] + f_q[0])],
           [0, r'$K/\alpha(0)$'])
plt.xlim([-50, 1300]); plt.ylim([-9, 105])
plt.legend(fontsize=9, ncol=1, frameon=False)

plt.savefig('Fig2.pdf', format='pdf', bbox_inches='tight',
            pad_inches=0, transparent=False)
plt.show()
